# ***Libraries, Tools and Definitions***

In [ ]:
import ebooklib
import re
import requests
import os
import json
import statistics
import textwrap
import numpy as np

from ebooklib import epub
from bs4 import BeautifulSoup
from lxml import html
from tqdm import tqdm
from collections import Counter

Choose the ***target folder*** and the ***journal*** name depending on the journal you wish to download files from

In [ ]:
parent_path = '../MDPI Articles/' # This is the path of the parent directory where all MDPI articles will be stored

#target_folder = 'MAKE 1996-2024/'
#target_folder = 'Applied Sci 1996-2024/'
#target_folder = 'Algo 1996-2024/'
#target_folder = 'Sensors 1996-2024/'
#target_folder = 'IJMS 1996-2024/'
target_folder = 'Sustainability 1996-2024/'

#journal = 'applsci'
#journal = 'ijms'
journal = 'sustainability'
#journal = 'make'
#journal = 'sensors'
#journal = 'algorithms'

.ris files contain important information about articles

In [ ]:
#ris_filename = 'MDPI_Applied_Sci_200_Articles_Page1_1996-2024.ris'
#ris_filename = 'MDPI_Algo_200_Articles_Page1_1996-2024.ris'
#ris_filename = 'MDPI_MAKE_200_Articles_Page1_1996-2024.ris'
#ris_filename = 'MDPI_sensors_200_Articles_Page1_1996-2024.ris'
#ris_filename = 'MDPI_ijms_200_Articles_Page1_1996-2024.ris'
ris_filename = 'MDPI_sustainability_200_Articles_Page1_1996-2024.ris'

Define the 'User Agent' as the MDPI server will not allow requests without a proper header

In [4]:
headers = {
        'User-Agent': 'Mozilla/5.0 (X11; Linux x86_64; rv:142.0) Gecko/20100101 Firefox/142.0',
        'referer': 'https://www.mdpi.com/'
}

# ***Helper Functions***

In [ ]:
"""
This function modifies a .ris file. A .ris file separates article information with a "ER -" 
"""

def modify_ris_file(file_path):
    # Read the content of the .ris file
    with open(file_path, 'r') as file:
        content = file.read()

    # Replace "ER  -" with "ER  -\n"
    modified_content = content.replace("ER  -", "ER  -\n")
    
    # Write the modified content back to the file
    with open(file_path, 'w') as file:
        file.write(modified_content)
    
    print(f"File {file_path} has been modified successfully.")

In [ ]:
"""
This function classifies articles into different categories based on the number of sections they contain and 
the presence of back matter sections commonly found in MDPI articles, such as author contributions, funding information, 
data availability statements, and references
"""

def classify_case(sections, back_sections):
    n_sections = len(sections)
    has_back = len(back_sections) > 0

    if n_sections == 0:
        return "No sections"
    
    elif n_sections == 1 and not has_back:
        return "1 section, no back"
    
    elif n_sections == 1 and has_back:
        return "1 section, with back"
    
    elif n_sections == 2 and not has_back:
        return "2 sections, no back"
    
    elif n_sections == 2 and has_back:
        return "2 sections, with back"
    
    elif n_sections >= 3 and not has_back:
        return "3+ sections, no back"
    
    elif n_sections >= 3 and has_back:
        return "3+ sections, with back"
    
    else:
        return "Other"  # Should never hit, just a safeguard

In [ ]:
def extract_metadata(file_path):
    titles = []      # List to store titles
    abstracts = []   # List to store abstracts
    dois = []        # List to store DOIs
    keywords_list = []  # List of lists, where each sub-list contains keywords for a paper
    
    # Open and read the file line-by-line
    with open(file_path, 'r') as file:

        keywords = []  # Temporary list for keywords of a single paper
        title, abstract, doi = "", "", ""

        for line in file:

            # Check for the title
            if line.startswith("TI  - "):
                title = line.replace("TI  - ", "").strip()

            # Check for the abstract
            elif line.startswith("AB  - "):
                abstract = line.replace("AB  - ", "").strip()

            # Check for the DOI
            elif line.startswith("DO  - "):
                doi = "https://doi.org/" + line.replace("DO  - ", "").strip()

            # Check for keywords (there can be multiple)
            elif line.startswith("KW  - "):
                keywords.append(line.replace("KW  - ", "").strip())
                
            # If we reach "ER  -", store the data for the current paper and reset variables
            elif line.startswith("ER  -"):
                # Store the extracted data
                titles.append(title)
                abstracts.append(abstract)
                dois.append(doi)
                keywords_list.append(keywords)
                
                # Reset for the next paper
                keywords = []
                title, abstract, doi = "", "", ""
    
    return titles, abstracts, dois, keywords_list

In [ ]:
"""
Extracts the text contained between two section titles from an EPUB document.

The function first iterates through all document items (chapters) in the EPUB,
parses their HTML content with BeautifulSoup, and collects the plain text from
each chapter. The text from all chapters is then merged into a single string
and split into individual lines.

To make matching more robust, consecutive whitespace characters are normalized
in both the target section title and the lines of the document. Once the line
corresponding to `section_title` is encountered, text extraction begins. The
extraction stops when a line matching `end_string` is found.

The extracted lines are joined into a single string, removing line breaks and
excess whitespace. If the specified section cannot be found, the function
returns None.

Parameters
----------
paper : epub.EpubBook
    EPUB document containing the article.
section_title : str
    Title of the section from which extraction should begin.
end_string : str
    Title of the section that marks the end of the extraction.

Returns
-------
str or None
    The extracted section text as a single string, or None if no content is
    extracted.
"""

def extract_section_text(paper, section_title, end_string):
    all_text = []

    # Extract text from each item (chapter) in the paper
    for item in paper.get_items():
        if item.get_type() == ebooklib.ITEM_DOCUMENT:
            # Parse the content with BeautifulSoup
            soup = BeautifulSoup(item.get_content(), 'html.parser')
            text = soup.get_text() # Represents the text content of a chapter/item
            all_text.append(text)

    # Combine all text into a single string
    # An element in all_text, which is essentially the text content of a chapter, may contain many lines that are separated with '\n'
    # The goal is to unify the text contents of each chapter and separate them with a '\n' 
    full_text = "\n".join(all_text) 

    # Split the full text of the article into lines
    # lines != all_text before we apply "\n".join(all_text) 
    lines = full_text.splitlines()

    # Normalize section titles (replace multiple spaces with a single space)
    section_title = re.sub(r'\s+', ' ', section_title.strip())
    end_string    = re.sub(r'\s+', ' ', end_string.strip())

    # Initialize flags
    extracting = False
    extracted_lines = []

    # Iterate through lines to extract the desired section
    for line in lines:
        normalized_line = re.sub(r'\s+', ' ', line.strip())  # Normalize spaces in the line

        if normalized_line == section_title:
            extracting = True  # Start extracting after finding the section title
            continue  # Skip the line corresponding to the section title

        if normalized_line == end_string:
            break  # Stop extracting when the end string is found

        if extracting:
            extracted_lines.append(line.strip())  # Collect the text while extracting

    # Join extracted lines without newlines to remove whitespace between them
    return ' '.join(extracted_lines).strip() if extracted_lines else None

## ***Web Scraping***

In [ ]:
# This function returns all the section titles of an article given its URL, and an XPath to find those section titles

def extract_title_from_xpath(url, xpath):
    # Fetch the content from the URL
    response = requests.get(url, headers=headers)
    response.raise_for_status()  # Raise an error for bad responses

    # Parse the HTML content
    tree = html.fromstring(response.content)

    # Extract the section title using the provided XPath
    elements = tree.xpath(xpath)
    
    # Get the title from the extracted elements
    titles = [element.text_content() for element in elements]

    return titles

In [ ]:
def scrape_and_download_file(url, relative_xpath, save_path, filename):    
    
    # Fetch the content from the URL
    response = requests.get(url, headers=headers)
    response.raise_for_status()  # Raise an error for bad responses

    # Parse the HTML content
    tree = html.fromstring(response.content)

    # Extract the file link using the provided relative XPath. We will use the file link to download the file
    file_links = tree.xpath(relative_xpath)

    # Check if there is a file link 
    if not file_links:
        print(f"No file link found. DOI: {url}. File: {filename}")
        return -2
    
    # Get the first file link (assuming one link)
    file_url = file_links[0].get('href')  # Assuming it's an <a> tag with href
    
    # Handle relative URLs by joining with base URL
    if not file_url.startswith('http'): # If the file url does not start with 'http'
        file_url = requests.compat.urljoin('https://www.mdpi.com', file_url)
    
    # Download the file
    file_response = requests.get(file_url, headers=headers)
    
    # Save the file to the specified path

    if file_response.status_code == 200: # If the file was downloaded successfully
        file_name = os.path.join(save_path, filename)
        file_name += '.epub'
        with open(file_name, 'wb') as f:
            f.write(file_response.content)
        
        return 1
    else: # If the file failed to download
        return -1

# ***Prepare and Explore Metadata***

In [ ]:
file_path = f'{parent_path}{ris_filename}' # Get the .ris file

In [ ]:
# Modify it 
modify_ris_file(file_path) # Only once needed for each unique .ris file

In [ ]:
# Retrieve article metadata from the .ris file
titles, abstracts, dois, keywords_list = extract_metadata(file_path)

# Output the extracted metadata
print("Titles:", titles)
print(len(titles))
print("Abstracts:", abstracts)
print(len(abstracts))
print("DOIs:", dois)
print(len(dois))
print("Keywords:", keywords_list)
print(len(keywords_list))

In [11]:
# We will be using article urls quite extensively so create a dictionary with all urls and their index
# All articles must have a url (doi). It's not possible for an article to not have a url (article doesn't exist in MDPI even though it's listed)
doi_dict = {
    doi: i for i, doi in enumerate(dois)
}

In [ ]:
doi_dict

In [ ]:
len(doi_dict)

In [ ]:
missing_titles    = [i      for i, title in enumerate(titles)       if title == ""]
missing_abstracts = [i      for i, abstract in enumerate(abstracts) if abstract == ""]
missing_dois      = [i      for i, doi in enumerate(dois)           if doi == ""]

In [ ]:
print(missing_titles); print(missing_abstracts); print(missing_dois)

In [ ]:
articles_without_keywords = []

for i, article_keywords in enumerate(keywords_list):
    
    if len(article_keywords) == 0: # If the list of keywords for an article is empty
        articles_without_keywords.append(i) 
    
    elif len(article_keywords) == 1: # If there is only one keyword for an article
        if article_keywords[-1] == "n/a": # and that keyword is named as "n/a"
            articles_without_keywords.append(i)

In [ ]:
for i in articles_without_keywords: print(i); print(keywords_list[i]); print(titles[i]); print(dois[i]); print(abstracts[i])

In [ ]:
# Redefine "dois" so that it doesn't include the urls which correspond to the articles without keywords
dois = [dois[i] 
        for i in range(len(dois)) 
        if i not in articles_without_keywords
        ]

In [ ]:
len(dois)

# ***Download MDPI Articles (epub format)***

## Pre-installation

In [ ]:
# Initially, downloaded_papers will be empty because Downloaded_Articles.txt is empty (we haven't downloaded any articles yet)

downloaded_papers = []

with open(f'{parent_path}{target_folder}Downloaded_Articles.txt', 'r') as file: # The Downloaded_Articles.txt file must already exist a-priori
    for line in file:
        paper_id, paper_doi = line.strip().split()
        downloaded_papers.append([paper_id, paper_doi])

downloaded_papers = np.array(downloaded_papers)

In [ ]:
len(downloaded_papers)

In [ ]:
downloaded_paper_dois = list(downloaded_papers[:, 1]) if len(downloaded_papers) != 0 else []
downloaded_paper_ids  = list(downloaded_papers[:, 0]) if len(downloaded_papers) != 0 else []

In [ ]:
# We are using the new "dois" (the one that doesn't include the urls which correspond to the articles without keywords)
# If we haven't downloaded any articles yet (len(downloaded_papers) == 0), we will use all urls
# Otherwise, we will use the urls that we haven't downloaded files from 

file_ids = list(range(len(dois))) if len(downloaded_papers) == 0 else list(set(list(range(len(dois)))) - set([int(paper_id) for paper_id in downloaded_paper_ids]))

In [ ]:
file_ids

In [ ]:
# Go inside the target folder and store the filenames of all the .epub files
epub_files = [file.split('.')[0] 
              for file in os.listdir(f"{parent_path}{target_folder}") 
              if file.endswith(".epub")]

In [ ]:
# CHECK: if the paper ids in the 'Downloaded_Articles.txt' file match with the filenames of all the .epub files that we have downloaded up until this point, then the process is correct 
for paper_id in downloaded_paper_ids:
    if paper_id not in epub_files:
        print(paper_id)

## Installation

In [ ]:
fails, successes = [], []
articles_without_epub = [] # For logging

# Get the urls that we have not yet downloaded articles from
remaining_papers = [doi 
                    for doi in dois # The most recent dois
                    if doi not in downloaded_paper_dois]

In [ ]:
save_path = f'{parent_path}{target_folder}'
epub_download_xpath = '//a[@id="epub_link"]' # Download link XPath for the article file

for i, url in tqdm(enumerate(remaining_papers), total=len(remaining_papers), desc="Downloading"):    # Scrape the page and download the file
    try:
        file_state = scrape_and_download_file(url, epub_download_xpath, save_path, str(file_ids[i]))
        if file_state == 1:
            successes.append((file_ids[i], url))
        else:
            if file_state == -1:
                print(f'Failed: Epub for article {file_ids[i]} not found. DOI: {url}')
                fails.append(url)
            else:
                fails.append(url)
                articles_without_epub.append(url)
        
    except:
        print(f'Error: Epub for article {file_ids[i]} not found. DOI: {url}')
        fails.append(url)

## Post-installation

In [ ]:
fails

In [ ]:
articles_without_epub

In [ ]:
successes

In [ ]:
with open(f'{parent_path}{target_folder}Downloaded_Articles.txt', 'a') as file:
    for paper_info in successes:
        file.write(f'{paper_info[0]} {paper_info[1]}\n')

In [ ]:
print(f'Number of available papers in epub format: {len(successes)}')
for paper_info in successes:
    print(f'{paper_info[0]} - {paper_info[1]}')

Below is the most important check in this process.
The .ris files are downloaded directly from the corresponding journal MDPI page. Each must contain information for 200 articles. An article must have a doi (url). I create a mapping for each doi and its corresponding index in the 'dois' list. So, the 'doi_dict' contains information from the original (no removed dois) .ris file. The 'downloaded_paper_dois' has all the dois (urls) that have been successfully downloaded. The below comparisson should output:
1) The dois from the articles without keywords and
2) The dois from the articles that failed to install

In [36]:
print(set(doi_dict.keys()) - set(downloaded_paper_dois)) 

{'https://doi.org/10.3390/su17010213', 'https://doi.org/10.3390/su17010212', 'https://doi.org/10.3390/su17010204', 'https://doi.org/10.3390/su17010118', 'https://doi.org/10.3390/su17010060'}


In [ ]:
# Redefine "dois" so that it doesn't include urls that have failed downloading
# The result so far is a list that 1) doesn't include article urls with no keywords 2) doesn't include article urls that have failed downloading 
dois = [doi 
        for doi in downloaded_paper_dois]

In [ ]:
len(dois)

In [ ]:
# Initially, "articles_without_epub" contained the article urls that have no epub format
# Now, the list contains the indices of those articles
articles_without_epub = [doi_dict[url] for url in articles_without_epub]

# ***Process MDPI Article Contents***

In [ ]:
article_keycontents = []
for url in tqdm(downloaded_paper_dois, total=len(downloaded_paper_dois), desc="Extracting"): # We use "downloaded_paper_dois" because it has all the urls that we have successfully downloaded up to this point

    try:
        # Fetch the content from the URL
        response = requests.get(url, headers=headers)
        response.raise_for_status()  # Raise an error for bad responses. We assume no error since we have the available papers

        # Parse the HTML content
        tree = html.fromstring(response.content)

        # ===== Original numbered sections =====
        section_number = 1
        article_sections = []
        while True:
            section = f'sec{section_number}-{journal}-'
            section_title_xpath = f'//*[contains(@id, "{section}")]/h2'

            # Extract the section title using the provided XPath
            elements = tree.xpath(section_title_xpath)
            
            # Get the title from the extracted elements
            title = [element.text_content() for element in elements]

            if not title:
                break
        
            article_sections.append(title[0])
            section_number += 1

        # ===== html-back content =====
        html_back_content = []

        # Sections with class "html-notes"
        notes_titles = tree.xpath('//div[@class="html-back"]/section[@class="html-notes"]/h2')
        html_back_content.extend([el.text_content().strip() for el in notes_titles])

        # Section with id "html-references-list"
        references = tree.xpath('//div[@class="html-back"]/section[@id="html-references_list"]/h2')
        html_back_content.extend([el.text_content().strip() for el in references])

        disclaimer_note = tree.xpath('//div[@class="html-back"]/section[@class="html-fn_group"]//div[@class="html-p"]/b')
        html_back_content.extend([el.text_content().strip() for el in disclaimer_note])

        article_keycontents.append((article_sections, html_back_content, url))

    except:
        article_keycontents.append('-')
        print(f'Bad response for article: {url}')

In [21]:
for i, keycontent  in enumerate(article_keycontents):
    if keycontent != '-':
        # Unpack tuple safely
        sections, html_back, url = keycontent

        first_sec = sections[0] if sections else "-"
        last_sec = sections[-1] if sections else "-"
        html_back_str = " | ".join(html_back) if html_back else "-"

        print(f"{i:<4} {downloaded_paper_ids[i]:<5} {first_sec:<50} {last_sec:<50} {html_back_str:<40} {url}")
    else:
        print(f"{i:<4} {downloaded_paper_ids[i]:<5} {'-':<50} {'-':<50} {'-':<40} {'-'}")


0    0       1. Introduction                                    6. Conclusions and Discussion                    Author Contributions | Funding | Data Availability Statement | Conflicts of Interest | References | Disclaimer/Publisher’s Note: https://doi.org/10.3390/su17010045
1    1       1. Introduction                                    5. Conclusions and Policy Recommendations        Author Contributions | Funding | Institutional Review Board Statement | Informed Consent Statement | Data Availability Statement | Conflicts of Interest | References | Disclaimer/Publisher’s Note: https://doi.org/10.3390/su17010043
2    2       1. Introduction                                    5. Conclusions                                   Author Contributions | Funding | Institutional Review Board Statement | Informed Consent Statement | Data Availability Statement | Conflicts of Interest | References | Disclaimer/Publisher’s Note: https://doi.org/10.3390/su17010047
3    3       1. Introduction     

In [22]:
# Check whether the urls in "downloaded_paper_dois" match with the urls in "article_keycontents"
for i in range(len(downloaded_paper_dois)):
    if downloaded_paper_dois[i] != article_keycontents[i][2]:
        print(f'{i} - {downloaded_paper_dois[i]} - {downloaded_paper_ids[i]}')

In [ ]:
len(article_keycontents)

## Create Datasets (JSON format)

In [ ]:
# Count cases across all articles
case_counts = Counter(
    classify_case(sections, back)
    for sections, back, _ in (ak for ak in article_keycontents)
)

# Print statistics
print("=== Section/Back Cases Statistics ===")
total_articles = len(article_keycontents)
for case, count in case_counts.items():
    print(f"{case:<25}: {count:>4} ({count/total_articles:.1%})")
    
print(f"Total articles: {total_articles}")


=== Section/Back Cases Statistics ===
3+ sections, with back   :  192 (98.5%)
3+ sections, no back     :    3 (1.5%)
Total articles: 195


In [ ]:
article_contents = []

for i, paper_id in tqdm(enumerate(downloaded_paper_ids), total=len(downloaded_paper_ids), desc="Extracting:"):
    sections, back, url = article_keycontents[i]

    # Skip papers with no sections or trivial cases
    if not sections:
        continue
    if not back and len(sections) == 1:
        continue

    paper = epub.read_epub(f"{parent_path}{target_folder}{paper_id}.epub")
    extracted_text = []

    # Back, at least 3 sections
    if back and len(sections) >= 3:
        extracted_text.append(extract_section_text(paper, sections[0], sections[1]))
        extracted_text.append(extract_section_text(paper, sections[-1], back[0]))

    # No back, at least 2 sections
    elif not back and len(sections) >= 2:
        extracted_text.append(extract_section_text(paper, sections[0], sections[1]))

    """ # Back, exactly 2 sections
    elif back and len(sections) == 2:
        extracted_text.append(extract_section_text(paper, sections[0], sections[-1]))
        extracted_text.append(extract_section_text(paper, sections[-1], back[0]))


    # Back, exactly 1 section
    elif back and len(sections) == 1:
        extracted_text.append(extract_section_text(paper, sections[0], back[0])) """

    # Store result or warn
    if extracted_text:
        text = abstracts[doi_dict[url]] + " " + " ".join(extracted_text)
        article_contents.append((text, url))
    else:
        print(f"Sections not found for article {paper_id}. DOI: {url}")


In [ ]:
len(article_contents)

In [ ]:
for content in article_contents[5:6]:
    print(f'\n{textwrap.fill(content[0], width=80)}')

In [ ]:
valid_indices = [doi_dict[a[1]] for a in article_contents] # doi_dict contains the original article urls

In [28]:
data = {
    'Titles':   [titles[i] for i in valid_indices],
    'Articles': [content[0] for content in article_contents],
    'Keywords': [keywords_list[i] for i in valid_indices]
}

# Save the dictionary to a JSON file
with open(f'{parent_path}{target_folder}MDPI_sustainability_Augmented_Content.json', 'w') as file:
    json.dump(data, file)

In [29]:
""" data = {
    'Titles':    [titles[i] for i in range(len(titles)) if i not in articles_without_keywords],
    'Abstracts': [abstracts[i] for i in range(len(abstracts)) if i not in articles_without_keywords],
    'Keywords':  [keywords_list[i] for i in range(len(keywords_list)) if i not in articles_without_keywords]
} """

data = {
    'Titles':    [titles[i] for i in valid_indices],
    'Abstracts': [abstracts[i] for i in valid_indices],
    'Keywords':  [keywords_list[i] for i in valid_indices]
}

# Save the dictionary to a JSON file
with open(f'{parent_path}{target_folder}MDPI_sustainability_Abstracts.json', 'w') as file:
    json.dump(data, file)

## Unify MDPI Articles in a JSON File

In [6]:
def unify_mdpi_files(parent_path, target_folders, article_data_files):
    all_titles, all_articles, all_keywords = [], [], []

    data_file_index = 0
    for folder in target_folders:
        file_path = os.path.join(parent_path, folder, article_data_files[data_file_index])
        
        with open(file_path, "r", encoding='utf-8') as file: # encoding='utf-8'
            data = json.load(file)

        all_titles.extend(data.get("Titles", []))
        all_articles.extend(data.get("Abstracts", [])) # You might want to change this in data.get("Abstracts", []) when using only abstracts
        all_keywords.extend(data.get("Keywords", []))

        data_file_index += 1

    return all_titles, all_articles, all_keywords

In [7]:
titles, articles, keywords_list = unify_mdpi_files('../MDPI Articles/', 
                                                   
                                                   ["MAKE 1996-2024/", 
                                                    "Applied Sci 1996-2024/", 
                                                    "Algo 1996-2024/", 
                                                    "IJMS 1996-2024/", 
                                                    "Sensors 1996-2024/", 
                                                    "Sustainability 1996-2024/"
                                                    ], 

                                                    ['MDPI_MAKE_Abstracts.json', 
                                                    'MDPI_Applied_Sci_Abstracts.json',
                                                    'MDPI_Algo_Abstracts.json',
                                                    'MDPI_ijms_Abstracts.json',
                                                    'MDPI_Sensors_Abstracts.json',
                                                    'MDPI_sustainability_Abstracts.json']
                                                    )

In [ ]:
"""['MDPI_MAKE_Augmented_Content.json', 
    'MDPI_Applied_Sci_Augmented_Content.json',
    'MDPI_Algo_Augmented_Content.json',
    'MDPI_ijms_Augmented_Content.json',
    'MDPI_Sensors_Augmented_Content.json',
    'MDPI_sustainability_Augmented_Content.json']"""

In [8]:
with open("UMDPI_Abstracts.json", "w", encoding="utf-8") as f:
    for i in range(len(articles)):
        json.dump({
            'name': str(i),
            'title': titles[i],
            'abstract': articles[i],
            'keywords': ';'.join(keywords_list[i])
        }, f, ensure_ascii=False)
        f.write("\n")

In [ ]:
from transformers import AutoTokenizer

# ------------------------------------------------------------------
# Configuration
# ------------------------------------------------------------------

jsonl_file = "data.jsonl"

model_name = "meta-llama/Meta-Llama-3-8B-Instruct"
auth_token = "YOUR_HF_TOKEN"  # Optional if model is public

tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    token=auth_token
)

# ------------------------------------------------------------------
# Read dataset and compute token lengths
# ------------------------------------------------------------------

document_lengths = []

with open(jsonl_file, "r", encoding="utf-8") as f:

    for line in f:

        j = json.loads(line)

        title = j["title"]
        abstract = j["abstract"]

        # Construct the document
        document = title + ". " + abstract

        # Count tokens (without BOS/EOS tokens)
        token_ids = tokenizer(
            document,
            add_special_tokens=False
        )["input_ids"]

        document_lengths.append(len(token_ids))

# ------------------------------------------------------------------
# Statistics
# ------------------------------------------------------------------

print(f"Number of documents : {len(document_lengths)}")
print(f"Minimum length      : {min(document_lengths)}")
print(f"Maximum length      : {max(document_lengths)}")
print(f"Mean length         : {statistics.mean(document_lengths):.2f}")
print(f"Median length       : {statistics.median(document_lengths):.2f}")
print(f"Std. deviation      : {statistics.stdev(document_lengths):.2f}")
print(f"25th percentile     : {statistics.quantiles(document_lengths, n=4)[0]:.2f}")
print(f"75th percentile     : {statistics.quantiles(document_lengths, n=4)[2]:.2f}")
print(f"90th percentile     : {statistics.quantiles(document_lengths, n=10)[8]:.2f}")
print(f"95th percentile     : {statistics.quantiles(document_lengths, n=20)[18]:.2f}")
print(f"99th percentile     : {statistics.quantiles(document_lengths, n=100)[98]:.2f}")